# Lab 4: Foreign Catalog over Google BigQuery — NYC Citi Bike

This lab practices **Lakehouse Federation**: querying an external database live through Unity Catalog, without copying data into Databricks. It's a personal lab for Unit 5, using Google BigQuery's public **NYC Citi Bike** dataset (`bigquery-public-data.new_york_citibike`) — bike-share trip data, a nice adjacent theme to the TTC transit data in Labs 1–3.

This is a **practice notebook**: each exercise explains the concept and gives you a task — write the SQL yourself in the `# TODO` cells.

## Prerequisites (set these up before running this notebook)

This lab needs a **Google Cloud service account** — set this up in the Google Cloud Console first:

1. Create (or use an existing) Google Cloud project.
2. Enable the **BigQuery API** for that project.
3. Create a **service account**, then generate and download a **JSON key** for it.
4. Grant the service account the **BigQuery User** and **BigQuery Data Viewer** roles (BigQuery User is required even for public datasets — it's what lets the service account run/bill queries and create the temporary materialization dataset).

Workspace/compute requirements:
- Unity Catalog enabled (default on new workspaces).
- A cluster on **Databricks Runtime 16.1+** with Standard or Dedicated access mode, **or** a Pro/Serverless SQL warehouse.
- `CREATE CONNECTION` privilege on the metastore (workspace/metastore admin has this by default).

## Store the service account key as a secret

Don't paste the JSON key directly into SQL. Store it as a Databricks secret first, from a terminal with the Databricks CLI configured:

```bash
databricks secrets create-scope bigquery-secrets
databricks secrets put-secret bigquery-secrets gcp-service-account-key --string-value "$(cat /path/to/your-key.json)"
```

(Or use Catalog Explorer's **Create a connection** wizard, which accepts the raw JSON directly and handles storage for you.)

## Exercise 1: Create the Connection

**Concept:** A connection is the top-level Unity Catalog object that stores the path and credentials for reaching an external system — it's created once and can back multiple foreign catalogs.

**Task:** Create a connection named `bigquery_public_data` of `TYPE bigquery`, using `GoogleServiceAccountKeyJson` sourced from the secret you stored above (`secret('bigquery-secrets', 'gcp-service-account-key')`).

> 🤖 **Genie Code tip:** *"Write a CREATE CONNECTION statement in Databricks SQL for a BigQuery connection authenticated with a service account key stored as a secret."*

In [ ]:
%sql
-- TODO: CREATE CONNECTION IF NOT EXISTS bigquery_public_data TYPE bigquery OPTIONS (GoogleServiceAccountKeyJson secret('bigquery-secrets', 'gcp-service-account-key'))

## Exercise 2: Create the Foreign Catalog

**Concept:** A foreign catalog maps a Unity Catalog catalog onto a database/project in the external system, using an existing connection. Point it at the `bigquery-public-data` project (not your own billing project) via the `dataProjectId` option, so it mirrors Google's public datasets — specifically `new_york_citibike`.

**Task:** Create a foreign catalog `bigquery_citibike` using the `bigquery_public_data` connection, with `dataProjectId 'bigquery-public-data'`. Then list schemas matching `*citibike*` to confirm it worked.

> 🤖 **Genie Code tip:** *"Write a CREATE FOREIGN CATALOG statement in Databricks SQL using an existing connection, with a dataProjectId option, then show schemas matching a pattern."*

In [ ]:
%sql
-- TODO: CREATE FOREIGN CATALOG IF NOT EXISTS bigquery_citibike USING CONNECTION bigquery_public_data OPTIONS (dataProjectId 'bigquery-public-data')

-- TODO: SHOW SCHEMAS IN bigquery_citibike LIKE '*citibike*'

## Exercise 3: Explore the Foreign Schema

**Concept:** Unity Catalog syncs metadata for foreign catalogs automatically, so you can browse a federated source the same way you browse a native one.

**Task:** List the tables in `bigquery_citibike.new_york_citibike`, then describe the structure of `citibike_stations`.

> 🤖 **Genie Code tip:** *"Write SQL to list tables in a schema and describe the columns of one specific table."*

In [ ]:
%sql
-- TODO: SHOW TABLES IN bigquery_citibike.new_york_citibike

-- TODO: DESCRIBE TABLE bigquery_citibike.new_york_citibike.citibike_stations

## Exercise 4: Query Foreign Tables — Pushdown vs. Materialization

**Concept:** Simple filters and column projections push down to BigQuery directly, no extra setup needed. Aggregations, joins, or sorting-with-limit need **materialization** enabled — Databricks writes a temporary table in BigQuery to compute the result there before streaming it back (small added BigQuery compute cost). `citibike_trips` has ~59 million rows, so always filter or aggregate — never `SELECT *` the whole table.

**Task:**
1. Query `citibike_stations` for stations with `capacity > 40`, selecting `station_id`, `name`, `latitude`, `longitude`, `capacity`, ordered by capacity descending, limited to 20 rows — this should push down without materialization.
2. Enable materialization with `SET spark.databricks.bigquery.enableMaterialization = true`.
3. Query `citibike_trips`, filtered to January 2017 (`starttime`), grouped by `start_station_name`, counting trips, ordered descending, top 15.

> 🤖 **Genie Code tip:** *"Write a filtered SELECT with ORDER BY and LIMIT against a foreign table, then show me how to enable BigQuery materialization and write a GROUP BY aggregation query with a date range filter."*

In [ ]:
%sql
-- TODO: SELECT station_id, name, latitude, longitude, capacity FROM bigquery_citibike.new_york_citibike.citibike_stations WHERE capacity > 40 ORDER BY capacity DESC LIMIT 20

In [ ]:
%sql
-- TODO: SET spark.databricks.bigquery.enableMaterialization = true

In [ ]:
%sql
-- TODO: SELECT start_station_name, COUNT(*) AS trip_count FROM bigquery_citibike.new_york_citibike.citibike_trips
-- WHERE starttime >= '2017-01-01' AND starttime < '2017-02-01'
-- GROUP BY start_station_name ORDER BY trip_count DESC LIMIT 15

## Exercise 5: Grant Access and Refresh Metadata

**Concept:** Foreign catalogs use the same Unity Catalog permission model as native catalogs — `USE CATALOG`/`USE SCHEMA` to browse, `SELECT` to query. Metadata is cached and can be force-refreshed if the source schema changes.

**Task:**
1. Grant `USE CATALOG` on `bigquery_citibike`, `USE SCHEMA` on `bigquery_citibike.new_york_citibike`, and `SELECT` on both `citibike_stations` and `citibike_trips` — to `` `account users` `` (swap in a real group to actually test).
2. Run `REFRESH FOREIGN CATALOG` on `bigquery_citibike`.

> 🤖 **Genie Code tip:** *"Write GRANT statements for USE CATALOG, USE SCHEMA, and SELECT on a foreign catalog's tables, then a REFRESH FOREIGN CATALOG statement."*

In [ ]:
%sql
-- TODO: GRANT USE CATALOG ON CATALOG bigquery_citibike TO `account users`
-- TODO: GRANT USE SCHEMA ON SCHEMA bigquery_citibike.new_york_citibike TO `account users`
-- TODO: GRANT SELECT ON TABLE bigquery_citibike.new_york_citibike.citibike_stations TO `account users`
-- TODO: GRANT SELECT ON TABLE bigquery_citibike.new_york_citibike.citibike_trips TO `account users`

-- TODO: REFRESH FOREIGN CATALOG bigquery_citibike

## Recap

By the end of this notebook you should have practiced: creating a BigQuery connection with secret-backed auth, creating a foreign catalog with `dataProjectId`, exploring foreign schema/table metadata, a pushdown query, enabling materialization for an aggregation query, granting permissions on a foreign catalog, and refreshing foreign metadata.

Every query here runs against live BigQuery data — nothing is copied into Databricks, and Unity Catalog governs and audits access the whole way through, exactly like it does for native tables.

Next: **Lab 5** (AI/BI Genie space) — builds a natural-language interface over the tables from Lab 3.